# Next Word Prediction Using LSTM and GRU
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1. **Data Collection**: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.
2. **Data Preprocessing**: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.
3. **Model Building**: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.
4. **Model Training**: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.
5. **Model Evaluation**: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.
6. **Deployment**: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [ ]:
import nltk
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


True

## Import libraries

In [ ]:
# Essential libraries
import pandas as pd
import numpy as np
import pickle

# NLP libraries
from nltk.corpus import gutenberg  # text document from Gutenberg project

# TensorFlow libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, GRU, Dropout, Input
from tensorflow.keras.models import load_model


# Scikit-learn libraries
from sklearn.model_selection import train_test_split as TTS

## Load dataset and saved in local

In [ ]:
# Load data
data = gutenberg.raw('shakespeare-hamlet.txt')

# Save data into local
with open("data/hamlet.txt", "w", encoding='utf-8') as file:
    file.write(data)

## Data preprocessing

### Load dataet

In [ ]:
# Load dataset
url = '/content/hamlet.txt'
with open(url, 'r') as file:
    text = file.read()

print(text)

[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not a Mouse stirring

   Barn. Well, goodnight. If you do meet Horatio and
Marcellus, the Riuals of my Watch, bid them make hast.
Enter Horatio and Marcellus.

  Fran. I thinke I heare them. Stand: who's there?
  Hor. Friends to this ground

   Mar. And Leige-men to the Dane

   Fran. Giue you good night

   Mar. O farwel honest Soldier, who hath relieu'd you?
  Fra. Barnardo ha's my place: giue you goodnight.

Exit Fran.

  Mar. Holla Barnardo

   Bar. Say, what is Horatio there?
  Hor. A peece of

### Tokenize text

```python
text = "the food is good the food is delicious"
```

The tokenizer builds a vocabulary similar to this:

Word        | Frequency | Index |
------------| ----------|-------|
the         |    2      |   1   |
food        |    2      |   2   |
is          |    1      |   3   |
good        |    1      |   4   |
delocious   |    1      |   5   |


In [ ]:
tokenizer = Tokenizer()                    # converts text to lowercase and removes certain punctuation
tokenizer.fit_on_texts([text])             # analyzes the input text and builds a vocabulary
TOTAL_WORDS = len(tokenizer.word_index)+1  # for padding index 0
TOTAL_WORDS

4818

In [ ]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'of': 4,
 'i': 5,
 'you': 6,
 'a': 7,
 'my': 8,
 'it': 9,
 'in': 10,
 'that': 11,
 'ham': 12,
 'is': 13,
 'not': 14,
 'his': 15,
 'this': 16,
 'with': 17,
 'your': 18,
 'but': 19,
 'for': 20,
 'me': 21,
 'lord': 22,
 'as': 23,
 'what': 24,
 'he': 25,
 'be': 26,
 'so': 27,
 'him': 28,
 'haue': 29,
 'king': 30,
 'will': 31,
 'no': 32,
 'our': 33,
 'we': 34,
 'on': 35,
 'are': 36,
 'if': 37,
 'all': 38,
 'then': 39,
 'shall': 40,
 'by': 41,
 'thou': 42,
 'come': 43,
 'or': 44,
 'hamlet': 45,
 'good': 46,
 'do': 47,
 'hor': 48,
 'her': 49,
 'let': 50,
 'now': 51,
 'thy': 52,
 'how': 53,
 'more': 54,
 'they': 55,
 'from': 56,
 'enter': 57,
 'at': 58,
 'was': 59,
 'oh': 60,
 'like': 61,
 'most': 62,
 'there': 63,
 'well': 64,
 'know': 65,
 'selfe': 66,
 'would': 67,
 'them': 68,
 'loue': 69,
 'may': 70,
 "'tis": 71,
 'vs': 72,
 'sir': 73,
 'qu': 74,
 'which': 75,
 'did': 76,
 'why': 77,
 'laer': 78,
 'giue': 79,
 'thee': 80,
 'ile': 81,
 'must': 82,
 'hath': 

### Create input sequences

In [ ]:
text.split('\n')

['[The Tragedie of Hamlet by William Shakespeare 1599]',
 '',
 '',
 'Actus Primus. Scoena Prima.',
 '',
 'Enter Barnardo and Francisco two Centinels.',
 '',
 "  Barnardo. Who's there?",
 '  Fran. Nay answer me: Stand & vnfold',
 'your selfe',
 '',
 '   Bar. Long liue the King',
 '',
 '   Fran. Barnardo?',
 '  Bar. He',
 '',
 '   Fran. You come most carefully vpon your houre',
 '',
 "   Bar. 'Tis now strook twelue, get thee to bed Francisco",
 '',
 "   Fran. For this releefe much thankes: 'Tis bitter cold,",
 'And I am sicke at heart',
 '',
 '   Barn. Haue you had quiet Guard?',
 '  Fran. Not a Mouse stirring',
 '',
 '   Barn. Well, goodnight. If you do meet Horatio and',
 'Marcellus, the Riuals of my Watch, bid them make hast.',
 'Enter Horatio and Marcellus.',
 '',
 "  Fran. I thinke I heare them. Stand: who's there?",
 '  Hor. Friends to this ground',
 '',
 '   Mar. And Leige-men to the Dane',
 '',
 '   Fran. Giue you good night',
 '',
 "   Mar. O farwel honest Soldier, who hath reli

In [ ]:
# Test code
txt = "Ham. Oh Villany! How? Let the doore be lock'd."

# Converts text into numerical sequences based on the tokenizer's word_index dictionary
tk_list = tokenizer.texts_to_sequences([txt])[0]
print("Token list:", tk_list)


# Create progressively longer sequences from the tokenized sentence
for i in range(1,len(tk_list)):
        print(tk_list[:i+1])

# NOTE: generatE training sequences for next-word prediction

Token list: [12, 60, 4728, 53, 50, 1, 868, 26, 4729]
[12, 60]
[12, 60, 4728]
[12, 60, 4728, 53]
[12, 60, 4728, 53, 50]
[12, 60, 4728, 53, 50, 1]
[12, 60, 4728, 53, 50, 1, 868]
[12, 60, 4728, 53, 50, 1, 868, 26]
[12, 60, 4728, 53, 50, 1, 868, 26, 4729]


In [ ]:
input_seq = []
for line in text.split('\n'):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        input_seq.append(tokens[:i+1])

input_seq

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [ ]:
max([len(x) for x in input_seq])

14

`map(function, iteration)`

In [ ]:
# Get Max words for pad sequences
MAX_SEQ = max(map(len, input_seq)) # same with "max([len(x) for x in input_seq])"
MAX_SEQ

14

In [ ]:
input_seq = np.array(pad_sequences(sequences=input_seq, maxlen=MAX_SEQ, padding='pre'))
input_seq.shape

(25732, 14)

### Prepare data for splitting

In [ ]:
# Sparate independent and dependent data
X = input_seq[:,:-1] # take all words, except last word, as independent feature
y = input_seq[:,-1]  # take last word as dependent feature

In [ ]:
X.shape, y.shape

((25732, 13), (25732,))

In [ ]:
y = to_categorical(x=y, num_classes=TOTAL_WORDS) # TOTAL_WORDS = 4818
y.shape

(25732, 4818)

In [ ]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

> The column needed to be predicted belongs `4818` features now.

In [ ]:
# Train-Test-Split data
X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((20585, 13), (5147, 13), (20585, 4818), (5147, 4818))

## Create model

In [55]:
# Define early stopping
early_stopping = EarlyStopping(monitor='val_accuracy',     # quantity to be monitored
                               patience=5,                 # number of epochs with no improvement after which training will be stopped
                               mode='max',
                               restore_best_weights=True)  # whether to restore model weights from the epoch with the best value of the monitored quantity

#### LSTM RNN model

- `TOTAL_WORDS` = 4818
- `MAX_SEQ` = 14

In [ ]:
# Define model (LSTM RNN)
lstm_model = Sequential()
lstm_model.add(Input(shape=(X_train.shape[1], )))  # each sample for input must have 13 values
lstm_model.add(Embedding(input_dim=TOTAL_WORDS,    # size of the vocabulary, including the reserved index for padding
                    output_dim=100,           # number of dimensions in each word's embedding vector
                    input_length=MAX_SEQ-1))  # intended length of the input sequence
lstm_model.add(LSTM(150, return_sequences=True))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(100))
lstm_model.add(Dense(TOTAL_WORDS, activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile model
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

#### GRU RNN

In [ ]:
# Define model (GRU RNN)
gru_model = Sequential()
gru_model.add(Input(shape=(X_train.shape[1], )))  # 13 input features
gru_model.add(Embedding(input_dim=TOTAL_WORDS,    # size of the vocabulary, including the reserved index for padding
                    output_dim=100,           # number of dimensions in each word's embedding vector
                    input_length=MAX_SEQ-1))  # intended length of the input sequence
gru_model.add(GRU(150, return_sequences=True))
gru_model.add(Dropout(0.2))
gru_model.add(GRU(100))
gru_model.add(Dense(TOTAL_WORDS, activation='softmax'))

In [ ]:
# Compile model
gru_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 13, 150)        │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 100)            │        75,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,157,418 (4.42 MB)

 Trainable params: 1,157,418 (4.42 MB)

 Non-trainable params: 0 (0.00 B)

## Train & Predict

In [ ]:
# Save tokenizer
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file, protocol=pickle.HIGHEST_PROTOCOL)

### LSTM RNN

#### Train model

In [56]:
history = lstm_model.fit(x=X_train,
                        y=y_train,
                        epochs=50,
                        validation_data=(X_test, y_test),
                        verbose=1,
                        callbacks=early_stopping)

# Save model
lstm_model.save("LSTM_RNN_model.h5")

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4770 - loss: 2.3952 - val_accuracy: 0.0546 - val_loss: 11.5341
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4832 - loss: 2.3617 - val_accuracy: 0.0556 - val_loss: 11.5715
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4916 - loss: 2.3288 - val_accuracy: 0.0556 - val_loss: 11.6504
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4973 - loss: 2.2921 - val_accuracy: 0.0542 - val_loss: 11.7009
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5028 - loss: 2.2679 - val_accuracy: 0.0554 - val_loss: 11.7562
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5066 - loss: 2.2332 - val_accuracy: 0.0542 - val_loss: 11.7831
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5146 - loss: 2.2074 - val_accuracy: 0.0523 - val_loss: 11.8639


#### Predict

In [62]:
# input_text = "To be or not to be"
input_text = "Enter Barnardo and Francisco two "
print(f"Input text:{input_text}")
max_seq = lstm_model.input_shape[1]
print("max_seq:", max_seq)

# Tokenize
tkn_lst = tokenizer.texts_to_sequences([input_text])[0]
print(tkn_lst)

pad_seq = pad_sequences(sequences=[tkn_lst], maxlen=max_seq)
print("pad_seq:", pad_seq)
print("pad_seq shape:", pad_seq.shape)

pred = lstm_model.predict(pad_seq, verbose=0)[0]
pred_word_index = np.argmax(pred)
print("pred_word_index:", pred_word_index)

for word, index in tokenizer.word_index.items():
    if index == pred_word_index:
        print("Next word:", word)

Input text:Enter Barnardo and Francisco two 
max_seq: 13
[57, 407, 2, 1181, 177]
pad_seq: [[   0    0    0    0    0    0    0    0   57  407    2 1181  177]]
pad_seq shape: (1, 13)
pred_word_index: 1892
Next word: centinels


### GRU RNN

#### Train model

In [59]:
history = gru_model.fit(x=X_train,
                        y=y_train,
                        epochs=50,
                        validation_data=(X_test, y_test),
                        verbose=1,
                        callbacks=early_stopping)

# Save model
gru_model.save("GRU_RNN_model.h5")

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7131 - loss: 1.2601 - val_accuracy: 0.0534 - val_loss: 10.5257
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7177 - loss: 1.2417 - val_accuracy: 0.0554 - val_loss: 10.5750
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7220 - loss: 1.2276 - val_accuracy: 0.0546 - val_loss: 10.6088
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7254 - loss: 1.2128 - val_accuracy: 0.0527 - val_loss: 10.6784
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7276 - loss: 1.1941 - val_accuracy: 0.0536 - val_loss: 10.7237


#### Predict

In [63]:
# input_text = "To be or not to be"
input_text = "Enter Barnardo and Francisco two "
print(f"Input text:{input_text}")
max_seq = gru_model.input_shape[1]
print("max_seq:", max_seq)

# Tokenize
tkn_lst = tokenizer.texts_to_sequences([input_text])[0]
print(tkn_lst)

pad_seq = pad_sequences(sequences=[tkn_lst], maxlen=max_seq)
print("pad_seq:", pad_seq)
print("pad_seq shape:", pad_seq.shape)

pred = gru_model.predict(pad_seq, verbose=0)[0]
pred_word_index = np.argmax(pred)
print("pred_word_index:", pred_word_index)

for word, index in tokenizer.word_index.items():
    if index == pred_word_index:
        print("Next word:", word)

Input text:Enter Barnardo and Francisco two 
max_seq: 13
[57, 407, 2, 1181, 177]
pad_seq: [[   0    0    0    0    0    0    0    0   57  407    2 1181  177]]
pad_seq shape: (1, 13)
pred_word_index: 1892
Next word: centinels


## Load models

In [ ]:
# Load tokenizer
with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

In [ ]:
# Load models
lstm = load_model('/content/LSTM_RNN_model.h5')
gru = load_model('/content/GRU_RNN_model.h5')

### Helper function

In [ ]:
def predict_next_word(model, tokenizer, text):
    max_seq = model.input_shape[1]
    tkn_lst = tokenizer.texts_to_sequences([text])[0]
    pad_seq = pad_sequences(sequences=[tkn_lst], maxlen=max_seq)
    pred = model.predict(pad_seq, verbose=0)[0]
    pred_word_index = np.argmax(pred)

    return tokenizer.index_word.get(pred_word_index, None)

In [ ]:
# Using LSTM RNN model
text = "To be or not to be"
next_word = predict_next_word(lstm, tokenizer, text)
print(f"Input text: {text}")
print(f"Next predicted word: {next_word}")

Input text: To be or not to be
Next predicted word: the


In [ ]:
# Using GRU RNN model
text = "To be or not to be"
next_word = predict_next_word(gru, tokenizer, text)
print(f"Input text: {text}")
print(f"Next predicted word: {next_word}")

Input text:   Barn. Last night of all,When yond same
Next predicted word: None
